# Directional locomotion (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/ms3/tutorials/5.4_Fullbody_Directional_Locomotion.ipynb)

Train / load a muscle-driven full-body policy that walks in any of 8 compass
directions (N / NE / E / SE / S / SW / W / NW).

This tutorial **trains** a heading-conditioned student (`ActorCritic`) by
behavior-cloning the public MuscleMimic teacher `amathislab/mm-10m-2`.

**Colab `QUICK_MODE` (default):** small teacher dataset + a short training
loop, then eval/render the **student you just trained**. That student will
not match the published 350k / 120-epoch numbers (`survival@200 = 72.5%`).
Set `MYOSUITE_FULL_BC=1` (or `QUICK_MODE = False`) for the full collection.

1. Install MyoSuite + MuscleMimic extras + `orbax-checkpoint`
2. Download two circular walking clips
3. Roll the teacher and record `(obs, action)` pairs
4. Train `ActorCritic` (528 → 354) and save `policy_bc_best.pt`
5. Evaluate and render **that checkpoint**

Runtime: **CPU is enough**. Collection is the slow step.

## 0 — Colab / local install

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# amathislab/mm-10m-2's checkpoint sharding metadata pins a CPU device
# ("TFRT_CPU_0"); jax.local_devices() must expose exactly that device name for
# Orbax restore to succeed. If jax-cuda12-plugin is installed and a GPU is
# present, jax.local_devices() reports a CudaDevice instead and restore fails
# with "ValueError: Device TFRT_CPU_0 was not found in jax.local_devices()".
# Force CPU regardless of what's installed -- matches this notebook's own
# "Runtime: CPU is enough" design intent (see intro cell).
os.environ.setdefault("JAX_PLATFORMS", "cpu")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

QUICK_MODE = os.environ.get("MYOSUITE_FULL_BC", "0") != "1"
COLLECT_TEACHER = True
TRAIN_IF_MISSING = True

# Optional: Hugging Face file with a flat ActorCritic state_dict.
# Example: "owner/repo" + filename "policy_bc_best.pt"
CKPT_HF_REPO = os.environ.get("BC_DIRECTIONAL_CKPT_REPO", "").strip()
CKPT_HF_FILE = os.environ.get("BC_DIRECTIONAL_CKPT_FILE", "policy_bc_best.pt").strip()

# Public install source used when MyoSuite is not already on sys.path.
GIT_URL = os.environ.get(
    "MYOSUITE_GIT_URL",
    "https://github.com/MyoHub/myosuite.git",
)
# ms3 is the public branch that ships MuscleMimicFullbodyDirectionalEnv.
GIT_REF = os.environ.get("MYOSUITE_GIT_REF", "ms3")


def _has_myosuite_pkg() -> bool:
    try:
        import myosuite  # noqa: F401
        from myosuite.envs.myo.tasks.mimic.cpu import (  # noqa: F401
            MuscleMimicFullbodyDirectionalEnv,
        )
        return True
    except Exception:
        return False


def _find_repo_root() -> Path | None:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "myosuite" / "__init__.py").is_file():
            return d
    return None


if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "libegl1", "libgles2"],
        check=False,
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "huggingface_hub>=0.20",
            "torch",
            "imageio[ffmpeg]",
            "mediapy",
            "scipy",
            "orbax-checkpoint>=0.11.22,<0.11.24",
            "flax",
        ]
    )

repo = _find_repo_root()
if repo is not None:
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    if not _has_myosuite_pkg():
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo}[musclemimic]"]
        )
elif not _has_myosuite_pkg():
    dest = Path("/content/myosuite") if IN_COLAB else Path.cwd() / "myosuite_src"
    if not dest.exists():
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(dest)]
        )
    if IN_COLAB:
        os.chdir(dest)
    if str(dest) not in sys.path:
        sys.path.insert(0, str(dest))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{dest}[musclemimic]"]
    )
    repo = dest

if COLLECT_TEACHER:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "orbax-checkpoint>=0.11.22,<0.11.24", "flax"]
    )

print("IN_COLAB:", IN_COLAB, " QUICK_MODE:", QUICK_MODE)
print("cwd:", Path.cwd())

In [ ]:
import math
from pathlib import Path

import mujoco
import numpy as np
import torch
from IPython.display import Video, display

_here = Path.cwd()
if (_here / "myosuite").exists():
    REPO_ROOT = _here
elif (_here.parent / "myosuite").exists():
    REPO_ROOT = _here.parent
else:
    REPO_ROOT = _here
DATA_NPZ = REPO_ROOT / "runs" / "bc_directional_v2" / "bc_directional.npz"
CKPT_DIR = REPO_ROOT / "runs" / "bc_directional_v2"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RENDER_DIR = REPO_ROOT / "renders"
RENDER_DIR.mkdir(exist_ok=True)

OBS_DIM = 528
ACT_DIM = 354

N_SEEDS = 1 if QUICK_MODE else 10
N_STEPS = 80 if QUICK_MODE else 200
EVAL_LABELS = ["N", "E"] if QUICK_MODE else ["E", "NE", "N", "NW", "W", "SW", "S", "SE"]
RENDER_LABELS = ["N", "E"] if QUICK_MODE else ["E", "NE", "N", "NW", "W", "SW", "S", "SE"]
N_EPS_1V1 = 1 if QUICK_MODE else 2
STEPS_1V1 = 200 if QUICK_MODE else 2000

CKPT = next(
    (
        REPO_ROOT / "runs" / d / "policy_bc_best.pt"
        for d in ("bc_directional_v2", "bc_directional_v3", "bc_directional_v1")
        if (REPO_ROOT / "runs" / d / "policy_bc_best.pt").exists()
    ),
    CKPT_DIR / "policy_bc_best.pt",
)

if CKPT_HF_REPO and not CKPT.exists():
    from huggingface_hub import hf_hub_download

    CKPT = Path(
        hf_hub_download(repo_id=CKPT_HF_REPO, filename=CKPT_HF_FILE)
    )
    print("Downloaded checkpoint:", CKPT)

print("Repo root:", REPO_ROOT)
print("Checkpoint:", CKPT, "exists=", CKPT.exists())
print("Eval:", EVAL_LABELS, "x", N_SEEDS, "seeds x", N_STEPS, "steps")

## 1 — Motion clip selection

A clockwise or counter-clockwise walking circle sweeps every compass heading.
Picking the right **frame** initialises the agent already facing the target
direction — no separate straight-walk clip per sector.

| Clip | Subject | Typical sectors |
|------|---------|-----------------|
| `WalkInClockwiseCircle01` | 4 | NE, N, NW, W, S, SE |
| `WalkInCounterClockwiseCircle08` | 4 | E, SW |

In [ ]:
clips = {}
try:
    from huggingface_hub import hf_hub_download
    from myosuite.core.trajectory_io import load_motion_clip
    from myosuite.integrations.musclemimic.fullbody_model import (
        compile_mimic_fullbody_mjmodel,
        default_mimic_fullbody_config,
    )

    GAIT_REPO = "amathislab/musclemimic-retargeted"
    SUBJECT = "4"

    cfg = default_mimic_fullbody_config()
    model, _, _ = compile_mimic_fullbody_mjmodel(cfg)

    CLIP_NAMES = [
        "WalkInClockwiseCircle01",
        "WalkInCounterClockwiseCircle08",
    ]

    for name in CLIP_NAMES:
        hf_path = hf_hub_download(
            repo_id=GAIT_REPO,
            filename=f"MyoFullBody/gmr/KIT/{SUBJECT}/{name}_poses.npz",
            repo_type="dataset",
        )
        clips[name] = load_motion_clip(
            Path(hf_path),
            expected_nq=model.nq,
            expected_nv=model.nv,
        )
        print(f"{name}: {len(clips[name].qpos)} frames")
    print("Clips loaded. nq/nv/nu=", model.nq, model.nv, model.nu)
except Exception as e:
    print(f"[SKIP] Full-body clips / model not available ({type(e).__name__}: {e})")
    print("Install pip install -e '.[musclemimic]' and download public clips from")
    print("https://huggingface.co/datasets/amathislab/musclemimic-retargeted")


In [ ]:
SECTOR_DEG = list(range(0, 360, 45))
LABELS = ["E", "NE", "N", "NW", "W", "SW", "S", "SE"]
sector_assignments = {}

if not clips:
    print("[SKIP] No clips loaded; sector assignment needs the Hugging Face walking circles.")
else:
    _data_ref = mujoco.MjData(model)

    def clip_walk_angle_deg(clip, frame: int) -> float:
        _data_ref.qpos[:] = clip.qpos[frame]
        _data_ref.qvel[:] = clip.qvel[frame]
        mujoco.mj_forward(model, _data_ref)
        return math.degrees(math.atan2(float(_data_ref.qvel[1]), float(_data_ref.qvel[0])))

    def angular_error_deg(a: float, b: float) -> float:
        return abs((a - b + 180) % 360 - 180)

    print(f"{'Sector':>6}  {'Best clip':45}  {'Frame':>5}  {'Error deg':>9}")
    for deg, label in zip(SECTOR_DEG, LABELS):
        best_err, best_clip, best_frame = float("inf"), "", 0
        for name, clip in clips.items():
            for frame in range(0, len(clip.qpos), 5):
                err = angular_error_deg(clip_walk_angle_deg(clip, frame), float(deg))
                if err < best_err:
                    best_err, best_clip, best_frame = err, name, frame
        sector_assignments[label] = (best_clip, best_frame, best_err)
        print(f"{label:>6}  {best_clip:45}  {best_frame:>5}  {best_err:>9.2f}")


## 2 — Observation + teacher BC collection

**528-dim obs:** `qpos[7:]` (82) + `qvel[6:]` (82) + `act` (354) +
body-frame root XY vel (2) + heading `[cos θ, sin θ]` (2) + orientation (6).

`heading_cmd` is `obs[520:522]` — after body-frame velocity at `[518:520]`.

This cell rolls `amathislab/mm-10m-2` and writes `bc_directional.npz`.
`QUICK_MODE` collects a few thousand transitions (minutes). Full BC is
350k transitions (hours).

In [ ]:
from scipy.spatial.transform import Rotation as R


def build_directional_obs(data: mujoco.MjData, theta_rad: float) -> np.ndarray:
    qpos = data.qpos.astype(np.float32)
    qvel = data.qvel.astype(np.float32)
    act = data.act.astype(np.float32)
    pelvis_quat = qpos[3:7]
    yaw = float(R.from_quat(pelvis_quat[[1, 2, 3, 0]]).as_euler("zyx")[0])
    c, s = math.cos(-yaw), math.sin(-yaw)
    vx_g, vy_g = float(qvel[0]), float(qvel[1])
    vel_body = np.array([c * vx_g - s * vy_g, s * vx_g + c * vy_g], dtype=np.float32)
    heading_cmd = np.array([math.cos(theta_rad), math.sin(theta_rad)], dtype=np.float32)
    w, x, y, z = pelvis_quat
    roll = math.atan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
    pitch = math.asin(float(np.clip(2 * (w * y - z * x), -1.0, 1.0)))
    wx_w, wy_w, wz_w = float(qvel[3]), float(qvel[4]), float(qvel[5])
    orientation = np.array(
        [roll, pitch, c * wx_w - s * wy_w, s * wx_w + c * wy_w, wz_w, float(qvel[2])],
        dtype=np.float32,
    )
    return np.concatenate([qpos[7:], qvel[6:], act, vel_body, heading_cmd, orientation])


print("obs_dim", build_directional_obs.__name__, "->", OBS_DIM)

In [ ]:
if DATA_NPZ.exists():
    d = np.load(DATA_NPZ)
    print(f"Dataset: {DATA_NPZ}  obs={d['obs'].shape}  actions={d['actions'].shape}")
elif COLLECT_TEACHER and clips:
    from huggingface_hub import snapshot_download
    from myosuite.integrations.musclemimic.bc_directional_collector import (
        BcCollectionConfig,
        collect_bc_dataset,
    )

    teacher_root = Path(snapshot_download(repo_id="amathislab/mm-10m-2"))
    n_per_clip = 4_000 if QUICK_MODE else 175_000
    dataset = collect_bc_dataset(
        teacher_root,
        model,
        list(clips.values()),
        BcCollectionConfig(
            samples_per_clip=n_per_clip,
            episode_len=80 if QUICK_MODE else 400,
            seed=0,
        ),
    )
    DATA_NPZ.parent.mkdir(parents=True, exist_ok=True)
    np.savez(DATA_NPZ, obs=dataset["obs"], actions=dataset["actions"], theta=dataset["theta"])
    print(f"Saved {dataset['obs'].shape[0]} transitions to {DATA_NPZ}")
else:
    print("Collection skipped: need COLLECT_TEACHER=True and loaded clips.")

## 3 — Policy architecture

6-layer MLP, SiLU + LayerNorm residual blocks, ~1.8M parameters:
`obs (528) → Linear(512) → SiLU → [LN+SiLU]×5 → actor_mean (354)`.

In [ ]:
try:
    from myosuite.envs.myo.tasks.mimic.policy import ActorCritic

    net = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
    print(f"Parameters: {sum(p.numel() for p in net.parameters()):}")
except Exception as e:
    print(f'[SKIP] ActorCritic / musclemimic policy not available ({type(e).__name__}: {e})')


## 4 — Train the student

Supervised MSE on the teacher dataset. Saves a flat `state_dict` to
`runs/bc_directional_v2/policy_bc_best.pt`. `QUICK_MODE` uses 20 epochs;
the published run was 120 epochs on 350k transitions (val MSE ≈ 0.00767).

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

if CKPT.exists():
    print(f"Checkpoint present — skip training: {CKPT}")
elif TRAIN_IF_MISSING and DATA_NPZ.exists():
    d = np.load(DATA_NPZ)
    obs_t = torch.as_tensor(d["obs"], dtype=torch.float32)
    act_t = torch.as_tensor(d["actions"], dtype=torch.float32)
    dataset = TensorDataset(obs_t, act_t)
    n_val = max(1, int(0.05 * len(dataset)))
    train_ds, val_ds = random_split(dataset, [len(dataset) - n_val, n_val])
    train_dl = DataLoader(train_ds, batch_size=256 if QUICK_MODE else 512, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=256 if QUICK_MODE else 512)
    policy = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
    optimiser = optim.Adam(policy.parameters(), lr=3e-4)
    EPOCHS = 20 if QUICK_MODE else 120
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)
    loss_fn = nn.MSELoss()
    best_val = float("inf")
    train_losses, val_losses = [], []
    for epoch in range(1, EPOCHS + 1):
        policy.train()
        running = 0.0
        for obs_b, act_b in train_dl:
            optimiser.zero_grad()
            pred, _ = policy(obs_b)
            loss = loss_fn(pred, act_b)
            loss.backward()
            optimiser.step()
            running += loss.item()
        scheduler.step()
        policy.eval()
        with torch.no_grad():
            val_loss = sum(loss_fn(policy(obs_b)[0], act_b).item() for obs_b, act_b in val_dl) / len(val_dl)
        train_losses.append(running / max(1, len(train_dl)))
        val_losses.append(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            torch.save(policy.state_dict(), CKPT)
        print(f"Epoch {epoch}/{EPOCHS}  train={train_losses[-1]:.5f}  val={val_loss:.5f}")
    print("Best val", best_val, "->", CKPT)
else:
    print("No checkpoint and training disabled. Eval/render will use a random policy (will fall).")

## 5 — Evaluation

`MuscleMimicFullbodyDirectionalEnv` / `myoFullBodyDirectional-v0`.
Published `bc_directional_v2` numbers (350k BC, 10×200, 8 dirs):
**survival@200 = 72.5%**, mean steps alive 190.6/200, alignment +0.698.

Colab `QUICK_MODE` uses fewer directions / seeds / steps so the cell finishes.

In [ ]:
try:
    from myosuite.envs.myo.tasks.mimic.cpu import MuscleMimicFullbodyDirectionalEnv

    eval_policy = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
    if CKPT.exists():
        eval_policy.load_state_dict(torch.load(CKPT, map_location="cpu", weights_only=True))
        print("Loaded", CKPT)
    else:
        print("Using untrained weights (demo only).")
    eval_policy.eval()

    label_to_deg = {lab: deg for lab, deg in zip(LABELS, SECTOR_DEG)}
    results = {}
    for label in EVAL_LABELS:
        theta = math.radians(float(label_to_deg[label]))
        survived, alignments, steps_alive = 0, [], []
        for seed in range(N_SEEDS):
            env = MuscleMimicFullbodyDirectionalEnv(
                seed=seed,
                target_speed=0.65,
                w_jpos=0.0,
                w_jvel=0.0,
                w_survival=0.3,
                angle_override=theta,
            )
            obs, _ = env.reset(seed=seed)
            raw = env.unwrapped
            if raw.model.na:
                raw.data.act[:] = 0.05
                mujoco.mj_forward(raw.model, raw.data)
                from myosuite.envs.gymnasium_env import CpuEnvAccessor

                raw._accessor = CpuEnvAccessor(raw.model, raw.data, raw._ctrl_dt)
                obs = raw._ensure_obs_gymnasium_compliant(
                    raw._obs_dict_to_vec(raw._get_obs_dict(raw._accessor))
                )
            start_xy = env.data.qpos[:2].copy()
            alive = True
            n_steps = 0
            for _ in range(N_STEPS):
                action = eval_policy.act(obs)
                obs, _, term, trunc, _ = env.step(action)
                n_steps += 1
                if term or trunc:
                    alive = False
                    break
            if alive:
                survived += 1
            steps_alive.append(n_steps)
            disp = env.data.qpos[:2].copy() - start_xy
            norm = float(np.linalg.norm(disp))
            alignments.append(
                float((disp / norm) @ np.array([math.cos(theta), math.sin(theta)])) if norm > 1e-4 else 0.0
            )
            env.close()
        results[label] = {
            "surv": survived / N_SEEDS,
            "align": float(np.mean(alignments)),
            "mean_steps_alive": float(np.mean(steps_alive)),
        }

    print(f"{'Dir':>4}  {'Surv':>7}  {'MeanSteps':>9}  {'Alignment':>9}")
    for label, v in results.items():
        print(f"{label:>4}  {v['surv']:>7.0%}  {v['mean_steps_alive']:>9.1f}  {v['align']:>+9.3f}")
except Exception as e:
    print(f'[SKIP] Full-body directional env not available ({type(e).__name__}: {e})')


## 6 — Render compass directions

Offscreen RGB frames, written as MP4 and shown inline.

In [ ]:
try:
    import os
    import sys

    import imageio

    if sys.platform != "darwin":
        os.environ["MUJOCO_GL"] = "egl"
        os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

    if not Path(CKPT).exists():
        raise RuntimeError(
            "No student checkpoint. Run the collect + train cells first "
            f"(expected {CKPT})."
        )
    eval_policy = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
    eval_policy.load_state_dict(torch.load(CKPT, map_location="cpu", weights_only=True))
    eval_policy.eval()
    print("Rendering student", CKPT)

    EIGHT_DIR_VIDEO = RENDER_DIR / ("dirs_quick.mp4" if QUICK_MODE else "bc_directional_8dirs.mp4")
    RENDER_W, RENDER_H, FPS = 320, 240, 40

    sector_frames: list[list[np.ndarray]] = []
    for label in RENDER_LABELS:
        theta = math.radians(float(label_to_deg[label]))
        env = MuscleMimicFullbodyDirectionalEnv(
            seed=2,
            target_speed=0.65,
            w_jpos=0.0,
            w_jvel=0.0,
            w_survival=0.3,
            angle_override=theta,
            render_mode="rgb_array",
        )
        obs, _ = env.reset(seed=2)
        raw = env.unwrapped
        if raw.model.na:
            raw.data.act[:] = 0.05
            mujoco.mj_forward(raw.model, raw.data)
            from myosuite.envs.gymnasium_env import CpuEnvAccessor

            raw._accessor = CpuEnvAccessor(raw.model, raw.data, raw._ctrl_dt)
            obs = raw._ensure_obs_gymnasium_compliant(
                raw._obs_dict_to_vec(raw._get_obs_dict(raw._accessor))
            )
        renderer = mujoco.Renderer(env.model, height=RENDER_H, width=RENDER_W)
        cam = mujoco.MjvCamera()
        cam.type = mujoco.mjtCamera.mjCAMERA_FREE
        cam.lookat[:] = [0.0, 0.0, 1.0]
        cam.distance = 3.5
        cam.azimuth = 135.0
        cam.elevation = -20.0
        frames: list[np.ndarray] = []
        for _ in range(N_STEPS):
            action = eval_policy.act(obs)
            obs, _, term, trunc, _ = env.step(action)
            renderer.update_scene(env.data, camera=cam)
            frames.append(renderer.render().copy())
            if term or trunc:
                break
        while len(frames) < N_STEPS:
            frames.append(frames[-1].copy())
        renderer.close()
        env.close()
        sector_frames.append(frames)
        print(label, len(frames), "frames")

    grid_frames = []
    n = len(sector_frames)
    cols = min(4, n)
    for t in range(N_STEPS):
        rows = []
        for r0 in range(0, n, cols):
            chunk = [sector_frames[i][t] for i in range(r0, min(r0 + cols, n))]
            while len(chunk) < cols:
                chunk.append(np.zeros_like(chunk[0]))
            rows.append(np.concatenate(chunk, axis=1))
        grid_frames.append(np.concatenate(rows, axis=0).astype(np.uint8))

    with imageio.get_writer(str(EIGHT_DIR_VIDEO), fps=FPS, quality=8, macro_block_size=1) as w:
        for f in grid_frames:
            w.append_data(f)
    print("Saved", EIGHT_DIR_VIDEO)
    display(Video(str(EIGHT_DIR_VIDEO), embed=True, html_attributes="controls loop"))
except Exception as e:
    print(f'[SKIP] Directional render not available ({type(e).__name__}: {e})')


## 7 — ChaseTag render

Prefer two-agent `myoChallengeChaseTagFBVs-v0` when it is registered.
On Colab / public `ms3` that id is often missing (registration is
optional); then use `myoChallengeChaseTagFBP2-v0` — one full-body agent
vs a scripted opponent, the env Gymnasium suggested.

There is no public `policy_bc_best.pt`. Without that file this cell loads
the MuscleMimic teacher `amathislab/mm-10m-2` and starts from a circular
walk clip so the humanoid actually locomotes. A random ActorCritic from a
standing reset will only collapse.

In [ ]:
from __future__ import annotations

import math
import os
import subprocess
import sys

# See setup cell: forces the CPU device Orbax restore needs regardless of
# whether jax-cuda12-plugin is installed.
os.environ.setdefault("JAX_PLATFORMS", "cpu")
if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
if "google.colab" in sys.modules:
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "libegl1", "libgles2", "libosmesa6"],
        check=False,
    )

import gymnasium as gym
import imageio
import mujoco
import myosuite
import myosuite.envs.myo.tasks.challenge  # noqa: F401
import numpy as np
import torch
from IPython.display import Video, display
from pathlib import Path

if hasattr(myosuite, "register_all_envs"):
    myosuite.register_all_envs()


def _gym_has(env_id: str) -> bool:
    try:
        gym.spec(env_id)
        return True
    except Exception:
        return False


ENV_1V1 = "myoChallengeChaseTagFBVs-v0"
ENV_P2 = "myoChallengeChaseTagFBP2-v0"
clips = globals().get("clips") or {}
OBS_DIM = int(globals().get("OBS_DIM", 528))
ACT_DIM = int(globals().get("ACT_DIM", 354))


teacher_runner = None
teacher_clip = None
teacher_model = None
teacher_data = None
controller_kind = "none"


def _ensure_clips():
    global clips
    if clips:
        return clips
    try:
        from huggingface_hub import hf_hub_download
        from myosuite.core.trajectory_io import load_motion_clip
        from myosuite.integrations.musclemimic.fullbody_model import (
            compile_mimic_fullbody_mjmodel,
            default_mimic_fullbody_config,
        )

        model, _, _ = compile_mimic_fullbody_mjmodel(default_mimic_fullbody_config())
        loaded = {}
        for name in ("WalkInClockwiseCircle01", "WalkInCounterClockwiseCircle08"):
            hf_path = hf_hub_download(
                repo_id="amathislab/musclemimic-retargeted",
                filename=f"MyoFullBody/gmr/KIT/4/{name}_poses.npz",
                repo_type="dataset",
            )
            loaded[name] = load_motion_clip(
                Path(hf_path), expected_nq=model.nq, expected_nv=model.nv
            )
            print(f"loaded clip {name}: {len(loaded[name].qpos)} frames")
        clips = loaded
    except Exception as exc:
        print(f"clip download failed ({type(exc).__name__}: {exc})")
        clips = {}
    return clips


def _ensure_teacher():
    global teacher_runner, teacher_clip, teacher_model, teacher_data
    if teacher_runner is not None:
        return teacher_runner
    if not _ensure_clips():
        return None
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "orbax-checkpoint>=0.11.22,<0.11.24", "flax"]
        )
        from myosuite.integrations.musclemimic.bc_directional_collector import (
            load_teacher_runner,
        )
        from myosuite.integrations.musclemimic.fullbody_checkpoint_io import (
            resolve_checkpoint_ref,
        )
        from myosuite.integrations.musclemimic.fullbody_model import (
            compile_mimic_fullbody_mjmodel,
            default_mimic_fullbody_config,
        )

        ref = resolve_checkpoint_ref("hf://amathislab/mm-10m-2")
        teacher_model, _, _ = compile_mimic_fullbody_mjmodel(default_mimic_fullbody_config())
        teacher_clip = next(iter(clips.values()))
        teacher_runner = load_teacher_runner(
            ref.local_path, teacher_model, teacher_clip, frame_skip=5
        )
        teacher_data = mujoco.MjData(teacher_model)
        teacher_runner.reset()
        print("Loaded MuscleMimic teacher amathislab/mm-10m-2")
        return teacher_runner
    except Exception as exc:
        print(f"teacher load failed ({type(exc).__name__}: {exc})")
        return None


def _ensure_eval_policy():
    global controller_kind
    pol = globals().get("eval_policy")
    ckpt = globals().get("CKPT")
    if ckpt is None:
        root = Path.cwd()
        ckpt = next(
            (
                root / "runs" / d / "policy_bc_best.pt"
                for d in ("bc_directional_v2", "bc_directional_v3", "bc_directional_v1")
                if (root / "runs" / d / "policy_bc_best.pt").exists()
            ),
            root / "runs" / "bc_directional_v2" / "policy_bc_best.pt",
        )
    ckpt = Path(ckpt)
    if pol is not None and ckpt.exists():
        controller_kind = "bc"
        return pol
    if ckpt.exists():
        from myosuite.envs.myo.tasks.mimic.policy import ActorCritic

        pol = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
        pol.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True))
        pol.eval()
        controller_kind = "bc"
        print("Loaded BC checkpoint", ckpt)
        return pol
    if _ensure_teacher() is not None:
        controller_kind = "teacher"
        return None
    from myosuite.envs.myo.tasks.mimic.policy import ActorCritic

    print("No BC checkpoint and teacher unavailable — random policy will fall.")
    controller_kind = "random"
    pol = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
    pol.eval()
    return pol


eval_policy = None
clips = _ensure_clips() or clips
chase_env_id = ENV_1V1 if _gym_has(ENV_1V1) else (ENV_P2 if _gym_has(ENV_P2) else "")
if chase_env_id == ENV_1V1 and not clips and _gym_has(ENV_P2):
    chase_env_id = ENV_P2
    print("No gait clips; using", ENV_P2)
print("ChaseTag env:", chase_env_id or "(none registered)")

RENDER_DIR = Path(globals().get("RENDER_DIR", Path.cwd() / "renders"))
RENDER_DIR.mkdir(parents=True, exist_ok=True)
N_EPS_1V1 = int(globals().get("N_EPS_1V1", 1))
STEPS_1V1 = int(globals().get("STEPS_1V1", 40 if "google.colab" in sys.modules else 80))

NQ_H, NV_H, NA_H = 89, 88, 354

if "build_directional_obs" not in globals():
    from scipy.spatial.transform import Rotation as R

    def build_directional_obs(data, theta_rad: float):
        qpos = np.asarray(data.qpos, dtype=np.float32)
        qvel = np.asarray(data.qvel, dtype=np.float32)
        act = np.asarray(data.act, dtype=np.float32)
        pelvis_quat = qpos[3:7]
        yaw = float(R.from_quat(pelvis_quat[[1, 2, 3, 0]]).as_euler("zyx")[0])
        c, s = math.cos(-yaw), math.sin(-yaw)
        vx_g, vy_g = float(qvel[0]), float(qvel[1])
        vel_body = np.array([c * vx_g - s * vy_g, s * vx_g + c * vy_g], dtype=np.float32)
        heading_cmd = np.array([math.cos(theta_rad), math.sin(theta_rad)], dtype=np.float32)
        w, x, y, z = pelvis_quat
        roll = math.atan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = math.asin(float(np.clip(2 * (w * y - z * x), -1.0, 1.0)))
        orientation = np.array(
            [roll, pitch, c * float(qvel[3]) - s * float(qvel[4]),
             s * float(qvel[3]) + c * float(qvel[4]), float(qvel[5]), float(qvel[2])],
            dtype=np.float32,
        )
        return np.concatenate([qpos[7:], qvel[6:], act, vel_body, heading_cmd, orientation])


def _obs_528(data, theta_rad: float) -> np.ndarray:
    from types import SimpleNamespace

    return build_directional_obs(
        SimpleNamespace(
            qpos=np.asarray(data.qpos[:NQ_H], dtype=np.float32),
            qvel=np.asarray(data.qvel[:NV_H], dtype=np.float32),
            act=np.asarray(data.act[:NA_H], dtype=np.float32),
        ),
        theta_rad,
    )


def _heading_to_xy(data, target_xy) -> float:
    dx = float(target_xy[0] - data.qpos[0])
    dy = float(target_xy[1] - data.qpos[1])
    return math.atan2(dy, dx)


def _make_renderer(model, height: int, width: int):
    import importlib

    last = None
    backends = ("glfw",) if sys.platform == "darwin" else ("egl", "osmesa")
    for backend in backends:
        os.environ["MUJOCO_GL"] = backend
        os.environ["PYOPENGL_PLATFORM"] = backend
        try:
            import mujoco.gl_context as glc

            importlib.reload(glc)
            mujoco.GLContext = glc.GLContext
            return mujoco.Renderer(model, height=height, width=width)
        except Exception as exc:
            last = exc
            print(f"Renderer backend {backend} failed: {type(exc).__name__}: {exc}")
    print("Continuing without OpenGL video.")
    return None


def _render_fbp2() -> None:
    env = gym.make(ENV_P2, render_mode="rgb_array")
    raw = env.unwrapped
    CHASETAG_DIR = RENDER_DIR / "chasetag_fbp2"
    CHASETAG_DIR.mkdir(parents=True, exist_ok=True)
    renderer = _make_renderer(raw.model, 360, 480)
    gait = teacher_clip or (next(iter(clips.values())) if clips else None)
    for ep in range(N_EPS_1V1):
        env.reset(seed=ep)
        frame_idx = 0
        if gait is not None:
            nq = min(NQ_H, int(gait.qpos.shape[1]), int(raw.model.nq))
            nv = min(NV_H, int(gait.qvel.shape[1]), int(raw.model.nv))
            raw.data.qpos[:nq] = gait.qpos[frame_idx, :nq]
            raw.data.qvel[:nv] = gait.qvel[frame_idx, :nv]
        raw.data.act[:NA_H] = 0.05
        mujoco.mj_forward(raw.model, raw.data)
        if teacher_runner is not None:
            teacher_runner.reset()
        frames = []
        last = 0
        print("controller:", controller_kind)
        for step in range(STEPS_1V1):
            last = step
            if teacher_runner is not None and gait is not None:
                nq = min(int(teacher_model.nq), int(raw.model.nq))
                nv = min(int(teacher_model.nv), int(raw.model.nv))
                teacher_data.qpos[:nq] = raw.data.qpos[:nq]
                teacher_data.qvel[:nv] = raw.data.qvel[:nv]
                if teacher_model.na and raw.model.na:
                    na = min(int(teacher_model.na), int(raw.model.na))
                    teacher_data.act[:na] = raw.data.act[:na]
                mujoco.mj_forward(teacher_model, teacher_data)
                frame_idx = min(frame_idx + 1, len(gait.qpos) - 1)
                action = np.clip(
                    teacher_runner.action_for(teacher_data, gait, frame_idx), 0.0, 1.0
                )
            else:
                opp = raw.opponent.get_opponent_pose()[:2]
                theta = _heading_to_xy(raw.data, opp)
                action = np.clip(eval_policy.act(_obs_528(raw.data, theta)), 0.0, 1.0)
            _, _, term, trunc, _ = env.step(action)
            if renderer is not None:
                cam = mujoco.MjvCamera()
                cam.type = mujoco.mjtCamera.mjCAMERA_FREE
                cam.lookat[:] = [float(raw.data.qpos[0]), float(raw.data.qpos[1]), 1.0]
                cam.distance = 5.0
                cam.azimuth = 135.0
                cam.elevation = -20.0
                renderer.update_scene(raw.data, camera=cam)
                frames.append(np.array(renderer.render(), dtype=np.uint8))
            if term or trunc:
                break
        print(f"ep{ep:02d} steps={last + 1} pelvis_z={float(raw.data.qpos[2]):.3f} ({ENV_P2})")
        if frames:
            out = CHASETAG_DIR / f"ep{ep:02d}_steps{last}.mp4"
            with imageio.get_writer(str(out), fps=50, quality=8, macro_block_size=1) as w:
                for f in frames:
                    w.append_data(f)
            print(f"saved {out}")
            display(Video(str(out), embed=True, html_attributes="controls loop"))
    if renderer is not None:
        renderer.close()
    env.close()


def _render_fbvs() -> None:
    _TINT = 0.55

    def _tint_agents(mj_model):
        a0 = np.array([0.85, 0.15, 0.15, 1.0])
        a1 = np.array([0.15, 0.35, 0.85, 1.0])
        for i in range(mj_model.ngeom):
            name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_GEOM, i) or ""
            orig = mj_model.geom_rgba[i].copy()
            if orig[3] < 0.01:
                continue
            if name.startswith("a0_") and not name.endswith("_floor"):
                mj_model.geom_rgba[i] = (1 - _TINT) * orig + _TINT * a0
                mj_model.geom_rgba[i, 3] = orig[3]
            elif name.startswith("a1_") and not name.endswith("_floor"):
                mj_model.geom_rgba[i] = (1 - _TINT) * orig + _TINT * a1
                mj_model.geom_rgba[i, 3] = orig[3]

    def _fix_floor(mj_model):
        for i in range(mj_model.ngeom):
            name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_GEOM, i) or ""
            if name in ("a0_floor", "a1_floor"):
                mj_model.geom_rgba[i, 3] = 0.0

    def _pelvis_yaw(qpos, adr):
        w, x, y, z = qpos[adr + 3], qpos[adr + 4], qpos[adr + 5], qpos[adr + 6]
        return float(np.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z)))

    def _bc_obs_for_agent(mj_model, data, meta, agent_id, heading_dir):
        jnt_ids = meta.jnt_ids[agent_id]
        root_qposadr = int(mj_model.jnt_qposadr[jnt_ids[0]])
        root_qveladr = int(mj_model.jnt_dofadr[jnt_ids[0]])
        local_qpos_start = int(mj_model.jnt_qposadr[jnt_ids[1]])
        local_qvel_start = int(mj_model.jnt_dofadr[jnt_ids[1]])
        nq_local = mj_model.nq // 2 - 7
        nv_local = mj_model.nv // 2 - 6
        local_qpos = data.qpos[local_qpos_start : local_qpos_start + nq_local].astype(np.float32)
        local_qvel = data.qvel[local_qvel_start : local_qvel_start + nv_local].astype(np.float32)
        act = data.act[meta.act_indices[agent_id]].astype(np.float32)
        yaw = _pelvis_yaw(data.qpos, root_qposadr)
        vx, vy = data.qvel[root_qveladr], data.qvel[root_qveladr + 1]
        c, s = math.cos(-yaw), math.sin(-yaw)
        vel_body = np.array([c * vx - s * vy, s * vx + c * vy], dtype=np.float32)
        rq = data.qpos[root_qposadr + 3 : root_qposadr + 7]
        w, x, y, z = rq
        roll = math.atan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = math.asin(float(np.clip(2 * (w * y - z * x), -1.0, 1.0)))
        wx_w, wy_w, wz_w = data.qvel[root_qveladr + 3 : root_qveladr + 6]
        orientation = np.array(
            [roll, pitch, c * wx_w - s * wy_w, s * wx_w + c * wy_w, wz_w, data.qvel[root_qveladr + 2]],
            dtype=np.float32,
        )
        return np.concatenate(
            [local_qpos, local_qvel, act, vel_body, heading_dir.astype(np.float32), orientation]
        )

    circ_list = list(clips.values()) if clips else []
    env_1v1 = gym.make(ENV_1V1)
    raw = env_1v1.unwrapped
    meta = raw._meta
    raw._config.fall_pelvis_z_threshold = 0.6
    _fix_floor(raw.model)
    _tint_agents(raw.model)
    CHASETAG_DIR = RENDER_DIR / "chasetag_1v1"
    CHASETAG_DIR.mkdir(parents=True, exist_ok=True)
    renderer_1v1 = _make_renderer(raw.model, 360, 480)
    trained = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    for ep in range(N_EPS_1V1):
        env_1v1.reset(seed=ep)
        h0 = trained[ep % 8]
        h1 = (h0 + math.pi) % (2 * math.pi)
        headings_used = {}
        for aid, angle in [("agent_0", h0), ("agent_1", h1)]:
            jnt_ids = meta.jnt_ids[aid]
            rqa = int(raw.model.jnt_qposadr[jnt_ids[0]])
            rva = int(raw.model.jnt_dofadr[jnt_ids[0]])
            root_xy = raw.data.qpos[rqa : rqa + 2].copy()
            raw.data.act[meta.act_indices[aid]] = 0.05
            if not circ_list:
                headings_used[aid] = np.array([math.cos(angle), math.sin(angle)], np.float32)
                continue
            best_clip, best_fi, best_err = None, 0, float("inf")
            for clip in circ_list:
                for fi in range(0, len(clip.qpos), 5):
                    qp = clip.qpos[fi]
                    yaw_f = float(
                        np.arctan2(2 * (qp[3] * qp[6] + qp[4] * qp[5]), 1 - 2 * (qp[5] ** 2 + qp[6] ** 2))
                    )
                    err = abs(math.atan2(math.sin(yaw_f - angle), math.cos(yaw_f - angle)))
                    if err < best_err:
                        best_err, best_clip, best_fi = err, clip, fi
            qp, qv = best_clip.qpos[best_fi], best_clip.qvel[best_fi]
            actual_yaw = float(
                np.arctan2(2 * (qp[3] * qp[6] + qp[4] * qp[5]), 1 - 2 * (qp[5] ** 2 + qp[6] ** 2))
            )
            for k, jid in enumerate(jnt_ids[1:]):
                raw.data.qpos[int(raw.model.jnt_qposadr[jid])] = qp[7 + k]
                raw.data.qvel[int(raw.model.jnt_dofadr[jid])] = qv[6 + k]
            raw.data.qpos[rqa : rqa + 2] = root_xy
            raw.data.qpos[rqa + 2] = qp[2]
            raw.data.qpos[rqa + 3 : rqa + 7] = qp[3:7]
            raw.data.qvel[rva : rva + 6] = qv[0:6]
            headings_used[aid] = np.array([math.cos(actual_yaw), math.sin(actual_yaw)], np.float32)
        mujoco.mj_forward(raw.model, raw.data)
        frames = []
        fell = {"agent_0": False, "agent_1": False}
        last = 0
        for step in range(STEPS_1V1):
            last = step
            for aid in ("agent_0", "agent_1"):
                if not fell[aid]:
                    obs = _bc_obs_for_agent(raw.model, raw.data, meta, aid, headings_used[aid])
                    raw.data.ctrl[meta.act_indices[aid]] = np.clip(eval_policy.act(obs), 0.0, 1.0)
                else:
                    raw.data.ctrl[meta.act_indices[aid]] = 0.0
            for _ in range(5):
                mujoco.mj_step(raw.model, raw.data)
            for aid in ("agent_0", "agent_1"):
                pz = raw.data.qpos[int(raw.model.jnt_qposadr[meta.jnt_ids[aid][0]]) + 2]
                if not fell[aid] and pz < 0.6:
                    fell[aid] = True
            cam = mujoco.MjvCamera()
            cam.type = mujoco.mjtCamera.mjCAMERA_FREE
            cam.lookat[:] = [0.0, 0.0, 1.0]
            cam.distance = 5.0
            cam.azimuth = 135.0
            cam.elevation = -20.0
            renderer_1v1.update_scene(raw.data, camera=cam)
            frames.append(np.array(renderer_1v1.render(), dtype=np.uint8))
            if all(fell.values()):
                break
        out = CHASETAG_DIR / f"ep{ep:02d}_steps{last}.mp4"
        with imageio.get_writer(str(out), fps=50, quality=8, macro_block_size=1) as w:
            for f in frames:
                w.append_data(f)
        print(f"ep{ep:02d} -> {out}")
        display(Video(str(out), embed=True, html_attributes="controls loop"))
    renderer_1v1.close()
    env_1v1.close()


try:
    eval_policy = _ensure_eval_policy()
    if controller_kind == "teacher" and _gym_has(ENV_P2):
        chase_env_id = ENV_P2
        print("Teacher demo uses", ENV_P2)
    if chase_env_id == ENV_1V1:
        _render_fbvs()
    elif chase_env_id == ENV_P2:
        _render_fbp2()
    else:
        print("[SKIP] No full-body ChaseTag env is registered.")
except Exception as e:
    print(f"[SKIP] ChaseTag render failed ({type(e).__name__}: {e})")


## Notes

- **This tutorial trains the student.** Run cells in order: clips → collect → train → eval.
- **Full BC:** `MYOSUITE_FULL_BC=1` (350k transitions, 120 epochs) or
  `scripts/modal_bc_directional_pipeline.py`.
- Validation MSE is a weak proxy for closed-loop survival — always run section 5.
- `heading_cmd` lives at `obs[520:522]`. Reset `data.act` to `0.05` every episode.
- ChaseTag: `myoChallengeChaseTagFBVs-v0` (1v1) when registered; else `myoChallengeChaseTagFBP2-v0`.
